[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/07_Parsing_and_Checker/Parsing_and_Checker_Deep_Dive.ipynb)

# 1.7 Parsing, Checker, and Shape Inference — Deep Dive

Three essential ONNX tools: the **parser** (concise text format), the **checker** (structural validation), and **shape inference** (propagating type and shape information through the graph).

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [The ONNX Text Format](#section-1) | A concise alternative to `make_*` functions |
| 2 | [Parser Syntax Deep Dive](#section-2) | Grammar rules and examples |
| 3 | [Parsing Complex Models](#section-3) | Multi-node, typed, and attributed models |
| 4 | [The Model Checker](#section-4) | What `check_model` validates |
| 5 | [Common Validation Errors](#section-5) | Error patterns and fixes |
| 6 | [Shape Inference](#section-6) | Propagating shapes through the graph |
| 7 | [Shape Inference Rules](#section-7) | Per-operator shape computation |
| 8 | [Practical Debugging Workflow](#section-8) | Using all three tools together |
| 9 | [Key Takeaways & Interview Questions](#section-9) | Summary |

### Prerequisites

- Completed **1.1–1.6** (graph construction, serialization, functions)
- Understanding of tensor shapes and types

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime numpy

In [ ]:
import numpy as np
import onnx
import onnx.parser
from onnx import TensorProto, shape_inference
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.checker import check_model

print(f'ONNX version: {onnx.__version__}')

<a id='section-1'></a>
## Section 1: The ONNX Text Format

### The Verbosity Problem

Building ONNX graphs with `make_*` functions is verbose. A simple linear regression requires ~15 lines of Python. For complex models, this becomes unwieldy.

### The Solution: `onnx.parser`

ONNX provides a **text format** that describes models in a compact, human-readable syntax:

```
make_* API (verbose):                    Text format (concise):

X = make_tensor_value_info(...)          <
A = make_tensor_value_info(...)            ir_version: 8,
B = make_tensor_value_info(...)            opset_import: ["" : 15]
Y = make_tensor_value_info(...)          >
n1 = make_node('MatMul',...)             agraph (float[I,J] X,
n2 = make_node('Add',...)                        float[I] A,
g = make_graph(...)                              float[I] B)
m = make_model(...)                        => (float[I] Y) {
                                             XA = MatMul(X, A)
~15 lines                                    Y = Add(XA, B)
                                           }
                                         ~8 lines
```

### Grammar Overview

```
Model   ::= '<' MetadataList '>' GraphDef
Metadata::= 'ir_version' ':' INT
          | 'opset_import' ':' '[' DomainVersionList ']'
Graph   ::= NAME '(' InputList ')' '=>' '(' OutputList ')' '{' NodeList '}'
Input   ::= TypeShape NAME
TypeShape::= 'float' '[' DimList ']' | 'int64' '[' DimList ']' | ...
Dim     ::= INT | NAME    (static or symbolic)
Node    ::= OutputNames '=' OpType '(' InputNames ')' AttrList?
```

<a id='section-2'></a>
## Section 2: Parser Syntax Deep Dive

### Basic Model

Let's parse a simple linear regression model.

In [ ]:
# Simple model: Y = MatMul(X, A) + B
text = '''
    <
        ir_version: 8,
        opset_import: [ "" : 15]
    >
    agraph (float[I,J] X, float[J] A, float[J] B) => (float[I] Y) {
        XA = MatMul(X, A)
        Y = Add(XA, B)
    }
    '''

model = onnx.parser.parse_model(text)
check_model(model)

print('Parsed model successfully!')
print(f'  Graph name: {model.graph.name}')
print(f'  IR version: {model.ir_version}')
print(f'  Nodes:      {len(model.graph.node)}')
print(f'  Inputs:     {[i.name for i in model.graph.input]}')
print(f'  Outputs:    {[o.name for o in model.graph.output]}')

# Verify shapes are preserved
for vi in model.graph.input:
    t = vi.type.tensor_type
    dims = [d.dim_param or str(d.dim_value) for d in t.shape.dim]
    print(f'  {vi.name}: shape={dims}')

In [ ]:
# Model with attributes
text_attrs = '''
    <
        ir_version: 8,
        opset_import: [ "" : 15]
    >
    transpose_graph (float[M,N] X) => (float[N,M] Y) {
        Y = Transpose <perm = [1, 0]>(X)
    }
    '''

model_t = onnx.parser.parse_model(text_attrs)
check_model(model_t)

node = model_t.graph.node[0]
print(f'Parsed Transpose with attributes:')
print(f'  op_type: {node.op_type}')
for a in node.attribute:
    print(f'  attr: {a.name} = {list(a.ints)}')

<a id='section-3'></a>
## Section 3: Parsing Complex Models

The text format supports multi-node graphs with various data types and operators.

In [ ]:
# Multi-layer perceptron: Y = Softmax(ReLU(X @ W1 + b1) @ W2 + b2)
mlp_text = '''
    <
        ir_version: 8,
        opset_import: [ "" : 15]
    >
    mlp (float[batch, 4] X,
         float[4, 8] W1, float[8] b1,
         float[8, 3] W2, float[3] b2)
    => (float[batch, 3] Y) {
        H1 = MatMul(X, W1)
        H2 = Add(H1, b1)
        H3 = Relu(H2)
        H4 = MatMul(H3, W2)
        H5 = Add(H4, b2)
        Y = Softmax <axis = 1>(H5)
    }
    '''

model_mlp = onnx.parser.parse_model(mlp_text)
check_model(model_mlp)

print('MLP model parsed successfully!')
print(f'\nComputation flow:')
for i, n in enumerate(model_mlp.graph.node):
    attrs = {a.name: a.i if a.type == 2 else list(a.ints)
             for a in n.attribute}
    print(f'  [{i}] {n.op_type:10s} {list(n.input):25s} → {list(n.output)}'
          f'{" " + str(attrs) if attrs else ""}')

<a id='section-4'></a>
## Section 4: The Model Checker

### What `check_model` Validates

`onnx.checker.check_model()` performs **structural validation** on a model. It catches errors that would otherwise cause runtime failures.

### Validation Categories

```
┌───────────────────────────────────────────────────────────────┐
│                check_model() Validation Pipeline              │
├───────────────────────────────────────────────────────────────┤
│                                                               │
│  1. IR VERSION CHECK                                          │
│     └─ Is ir_version valid and compatible with opset?         │
│                                                               │
│  2. OPSET VALIDATION                                          │
│     └─ Do all referenced opsets have valid domain + version?  │
│                                                               │
│  3. GRAPH STRUCTURE                                           │
│     ├─ Are all node op_types defined in the specified opset?  │
│     ├─ Do nodes respect topological ordering?                 │
│     └─ Are all consumed tensor names produced somewhere?      │
│                                                               │
│  4. TYPE CONSTRAINTS                                          │
│     ├─ Do inputs match operator schema type constraints?      │
│     └─ Are type variables (T) consistently bound?            │
│                                                               │
│  5. ATTRIBUTE VALIDATION                                      │
│     ├─ Are required attributes present?                       │
│     └─ Are attribute types correct?                          │
│                                                               │
│  6. INITIALIZER CHECK                                         │
│     └─ Do initializer names match graph names?               │
│                                                               │
└───────────────────────────────────────────────────────────────┘
```

<a id='section-5'></a>
## Section 5: Common Validation Errors

Let's trigger and diagnose common errors.

In [ ]:
# Error 1: Unknown operator
print('Error 1: Unknown operator')
print('-' * 50)
try:
    bad = onnx.parser.parse_model('''
        <ir_version: 8, opset_import: ["" : 15]>
        g (float[2,3] X) => (float[2,3] Y) {
            Y = FakeOperator(X)
        }''')
    check_model(bad)
except Exception as e:
    print(f'  Caught: {str(e)[:100]}')

# Error 2: Missing input
print('\nError 2: Undefined tensor name')
print('-' * 50)
try:
    X = make_tensor_value_info('X', TensorProto.FLOAT, [2, 3])
    Y = make_tensor_value_info('Y', TensorProto.FLOAT, [2, 3])
    node = make_node('Add', ['X', 'UNDEFINED'], ['Y'])
    graph = make_graph([node], 'bad', [X], [Y])
    model = make_model(graph, opset_imports=[make_opsetid('', 15)])
    check_model(model)
except Exception as e:
    print(f'  Caught: {str(e)[:100]}')

# Error 3: Type mismatch
print('\nError 3: Type mismatch (int + float)')
print('-' * 50)
try:
    bad3 = onnx.parser.parse_model('''
        <ir_version: 8, opset_import: ["" : 15]>
        g (float[2] X, int64[2] Y) => (float[2] Z) {
            Z = Add(X, Y)
        }''')
    check_model(bad3)
except Exception as e:
    print(f'  Caught: {str(e)[:100]}')

<a id='section-6'></a>
## Section 6: Shape Inference

### What Shape Inference Does

Shape inference propagates **type and shape information** from graph inputs through all intermediate tensors. Given input shapes and operator semantics, it computes the shape of every tensor in the graph.

### Formal Definition

For each operator $f$ with schema-defined shape inference rule $\mathcal{S}_f$:

$$\text{shape}(\text{output}) = \mathcal{S}_f(\text{shape}(\text{input}_1), \ldots, \text{shape}(\text{input}_k), \text{attributes})$$

### Why It Matters

| Benefit | Description |
|---------|-------------|
| **Memory planning** | Runtime can pre-allocate exact buffer sizes |
| **In-place ops** | Reuse buffers when shapes match |
| **Operator fusion** | Known shapes enable specialized fused kernels |
| **Validation** | Catch shape mismatches before runtime |
| **Debugging** | See intermediate tensor shapes for any model |

In [ ]:
# Demonstrate shape inference
text = '''
    <ir_version: 8, opset_import: ["" : 15]>
    inference_demo (float[batch, 4] X,
                    float[4, 8] W1,
                    float[8, 3] W2)
    => (float[batch, 3] Y) {
        H = MatMul(X, W1)
        H2 = Relu(H)
        Y = MatMul(H2, W2)
    }
    '''

model = onnx.parser.parse_model(text)

# Before shape inference: intermediate tensors have no shape info
print('Before shape inference:')
print(f'  value_info count: {len(model.graph.value_info)}')

# Run shape inference
inferred = shape_inference.infer_shapes(model)

print(f'\nAfter shape inference:')
print(f'  value_info count: {len(inferred.graph.value_info)}')

print(f'\nInferred intermediate tensor shapes:')
for vi in inferred.graph.value_info:
    t = vi.type.tensor_type
    dims = [d.dim_param or str(d.dim_value) for d in t.shape.dim]
    dtype = TensorProto.DataType.Name(t.elem_type)
    print(f'  {vi.name:10s}: {dtype} {dims}')

# Also show input/output shapes
print(f'\nFull shape map:')
for vi in list(inferred.graph.input) + list(inferred.graph.value_info) + list(inferred.graph.output):
    t = vi.type.tensor_type
    dims = [d.dim_param or str(d.dim_value) for d in t.shape.dim]
    print(f'  {vi.name:10s}: {dims}')

<a id='section-7'></a>
## Section 7: Shape Inference Rules

### Per-Operator Rules

Each operator has a specific shape inference function. Here are the key rules:

| Operator | Rule | Example |
|:---------|:-----|:--------|
| `MatMul` | $(m, k) \times (k, n) \to (m, n)$ | `[batch,4] × [4,8] → [batch,8]` |
| `Add` | Broadcasting: align from right | `[batch,8] + [8] → [batch,8]` |
| `Relu` | Identity: same shape as input | `[batch,8] → [batch,8]` |
| `Transpose` | Permute dims by `perm` | `[m,n] → [n,m]` (perm=[1,0]) |
| `Reshape` | Product of dims preserved | `[2,3,4] → [6,4]` |
| `Softmax` | Identity: same shape | `[batch,10] → [batch,10]` |
| `Conv` | $\lfloor (D + 2P - K) / S \rfloor + 1$ per spatial dim | Complex formula |

### MatMul Shape Inference

For tensors $A \in \mathbb{R}^{\ldots \times m \times k}$ and $B \in \mathbb{R}^{\ldots \times k \times n}$:

$$\text{shape}(\text{MatMul}(A, B)) = \text{broadcast}(\text{batch\_dims}) \times m \times n$$

where `batch_dims` follows NumPy broadcasting rules.

### Symbolic Dimension Propagation

When inputs use symbolic dimensions (e.g., `batch`), shape inference propagates them:

$$\text{If } X: [\text{batch}, 4] \text{ and } W: [4, 8] \text{ then } XW: [\text{batch}, 8]$$

This is a form of **symbolic execution** — the shapes are computed algebraically rather than numerically.

<a id='section-8'></a>
## Section 8: Practical Debugging Workflow

### The Three-Step Debugging Pipeline

```
Step 1: Parse / Build           Step 2: Check              Step 3: Shape Inference
┌──────────────────┐     ┌──────────────────┐      ┌──────────────────────────┐
│ Text or make_*   │────▶│ check_model()    │─────▶│ infer_shapes()           │
│ Create the model │     │ Structural valid │      │ Propagate shapes         │
└──────────────────┘     └──────────────────┘      └──────────────────────────┘
       │                        │                          │
       ▼                        ▼                          ▼
  Parse errors:           Schema errors:             Shape errors:
  - Syntax issues         - Unknown ops              - Dimension mismatch
  - Missing metadata      - Type mismatches          - Broadcasting failure
  - Invalid IR version    - Missing attributes       - Rank mismatch
```

In [ ]:
# Complete debugging workflow example
def debug_model(text_or_model, name='model'):
    """Apply the full parse → check → infer pipeline."""
    print(f'\n{"="*60}')
    print(f'Debugging: {name}')
    print(f'{"="*60}')

    # Step 1: Parse
    if isinstance(text_or_model, str):
        try:
            model = onnx.parser.parse_model(text_or_model)
            print(f'[1] Parse:    OK ({len(model.graph.node)} nodes)')
        except Exception as e:
            print(f'[1] Parse:    FAILED - {e}')
            return None
    else:
        model = text_or_model
        print(f'[1] Parse:    (pre-built model, {len(model.graph.node)} nodes)')

    # Step 2: Check
    try:
        check_model(model)
        print(f'[2] Check:    OK')
    except Exception as e:
        print(f'[2] Check:    FAILED - {str(e)[:80]}')
        return model

    # Step 3: Shape inference
    try:
        inferred = shape_inference.infer_shapes(model)
        n_inferred = len(inferred.graph.value_info)
        print(f'[3] Shapes:   OK ({n_inferred} intermediate shapes inferred)')

        for vi in inferred.graph.value_info:
            t = vi.type.tensor_type
            dims = [d.dim_param or str(d.dim_value) for d in t.shape.dim]
            print(f'     {vi.name}: {dims}')
    except Exception as e:
        print(f'[3] Shapes:   FAILED - {str(e)[:80]}')

    return inferred

# Test with a valid model
debug_model('''
    <ir_version: 8, opset_import: ["" : 15]>
    valid (float[N,4] X, float[4,2] W, float[2] b) => (float[N,2] Y) {
        H = MatMul(X, W)
        Y = Add(H, b)
    }''', 'Valid Linear Model')

# Test with a model that has shape issues
debug_model('''
    <ir_version: 8, opset_import: ["" : 15]>
    multi_step (float[batch,10] X, float[10,5] W1, float[5,3] W2)
    => (float[batch,3] Y) {
        H1 = MatMul(X, W1)
        H2 = Relu(H1)
        Y = MatMul(H2, W2)
    }''', 'Multi-Step MLP')

<a id='section-9'></a>
## Section 9: Key Takeaways & Interview Questions

### Summary

| Tool | Purpose | When to Use |
|:-----|:--------|:------------|
| `onnx.parser.parse_model()` | Convert text format to `ModelProto` | Quick prototyping, readable definitions |
| `onnx.checker.check_model()` | Structural validation | Always — after any model construction |
| `onnx.shape_inference.infer_shapes()` | Propagate shapes | Debugging, before optimization |

### Critical Points

1. **Parser syntax**: `<metadata> graph_name (typed_inputs) => (typed_outputs) { nodes }`
2. **Checker validates**: opset compatibility, operator schemas, type constraints, graph structure
3. **Shape inference is conservative**: unknown dims stay unknown, but symbolic dims propagate
4. **Debug pipeline**: Parse → Check → Infer shapes → Run inference

### Interview Questions

1. **Q**: What does shape inference compute and why is it important?
   - **A**: Shape inference propagates type and shape information from inputs through all intermediate tensors using per-operator rules. It enables memory pre-allocation, operator fusion, in-place operations, and early detection of shape mismatches.

2. **Q**: What are the main categories of errors caught by `check_model`?
   - **A**: IR version compatibility, opset validation, operator schema conformance (type constraints, required attributes), graph structure (DAG, all names defined), and initializer consistency.

3. **Q**: How does shape inference handle symbolic dimensions?
   - **A**: Symbolic dimensions (like `batch`) propagate algebraically. If input $X$ has shape `[batch, 4]` and weight $W$ has shape `[4, 8]`, MatMul infers output shape `[batch, 8]` — the symbolic dim is preserved.

---

**Next:** [Evaluation and Runtime](../08_Evaluation_and_Runtime/) — ReferenceEvaluator and ONNX Runtime.